# Content-Based Movie Recommendations

This notebook explores recommendations based on movie genres.

The collaborative-filtering model compared movies using patterns in
users' ratings. This model will compare movies using their attributes.

We will use the same training, validation, and test split and the same
candidate movies as the earlier experiments. This makes their
validation results comparable.

The test set will remain reserved for the final evaluation.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw" / "ml-latest-small"

movies = pd.read_csv(data_dir / "movies.csv")
ratings = pd.read_csv(data_dir / "ratings.csv")

ratings["rated_at"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s",
    utc=True
)

dates = ratings["rated_at"]

train_cutoff = dates.quantile(0.60)
validation_cutoff = dates.quantile(0.80)

train_ratings = ratings.loc[
    dates <= train_cutoff
].copy()

validation_ratings = ratings.loc[
    (dates > train_cutoff) & (dates <= validation_cutoff)
].copy()

test_ratings = ratings.loc[
    dates > validation_cutoff
].copy()

print("Training rows:", len(train_ratings))
print("Validation rows:", len(validation_ratings))
print("Test rows:", len(test_ratings))

Training rows: 60502
Validation rows: 20167
Test rows: 20167


Select a data set of movies with training ratings that includes the genre

In [2]:
content_movies = movies.loc[
    movies["movieId"].isin(train_ratings["movieId"]),
    ["movieId", "title", "genres"]
].copy()

content_movies = content_movies.sort_values(
    "movieId"
).reset_index(drop=True)

print("Candidate movies:", len(content_movies))

content_movies.head()

Candidate movies: 6233


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Create a multi-hot encoding data frame where genres are columns and movies are rows. A 1 means the movie belongs to that genre and a zero means it dfoesn't

In [3]:
def split_genres(genre_text):
    """Convert a movie's genre string into a list of genre labels."""
    if genre_text == "(no genres listed)":
        return []

    return genre_text.split("|")


content_movies["genre_list"] = content_movies["genres"].apply(
    split_genres
)

content_movies[["movieId", "title", "genre_list"]].head()

,movieId,title,genre_list
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]


Scikit-learn’s MultiLabelBinarizer converts these lists into a binary matrix.

In [4]:
from sklearn.preprocessing import MultiLabelBinarizer

genre_encoder = MultiLabelBinarizer()

genre_values = genre_encoder.fit_transform(
    content_movies["genre_list"]
)

genre_matrix = pd.DataFrame(
    genre_values,
    index=content_movies["movieId"],
    columns=genre_encoder.classes_
)

print("Genre matrix shape:", genre_matrix.shape)

genre_matrix.iloc[:5, :8]

Genre matrix shape: (6233, 19)


,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama
movieId,,,,,,,,
1,0,1,1,1,1,0,0,0
2,0,1,0,1,0,0,0,0
3,0,0,0,0,1,0,0,0
4,0,0,0,0,1,0,0,1
5,0,0,0,0,1,0,0,0


calculate genre similarity

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

toy_story_genres = genre_matrix.loc[[1]]
jumanji_genres = genre_matrix.loc[[2]]

genre_similarity = cosine_similarity(
    toy_story_genres,
    jumanji_genres
)[0, 0]

print("Toy Story genre count:", toy_story_genres.sum(axis=1).iloc[0])
print("Jumanji genre count:", jumanji_genres.sum(axis=1).iloc[0])
print("Genre cosine similarity:", genre_similarity)

Toy Story genre count: 5
Jumanji genre count: 3
Genre cosine similarity: 0.7745966692414834


Compare Toy Story with every candidate movie

In [6]:
seed_movie_id = 1

genre_similarity_scores = cosine_similarity(
    genre_matrix.loc[[seed_movie_id]],
    genre_matrix
)[0]

similar_genre_movies = pd.DataFrame({
    "movieId": genre_matrix.index,
    "similarity": genre_similarity_scores
}).merge(
    content_movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left",
    validate="one_to_one"
)

similar_genre_movies = similar_genre_movies.loc[
    similar_genre_movies["movieId"] != seed_movie_id
].sort_values(
    ["similarity", "movieId"],
    ascending=[False, True]
).reset_index(drop=True)

similar_genre_movies.head(10)

,movieId,similarity,title,genres
0,2294,1.000000,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
1,3114,1.000000,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
2,3754,1.000000,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
3,4016,1.000000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
4,4886,1.000000,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
5,45074,1.000000,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy
6,53121,1.000000,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy
7,673,0.912871,Space Jam (1996),Adventure|Animation|Children|Comedy|Fantasy|Sc...
8,4306,0.912871,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...
9,36397,0.912871,Valiant (2005),Adventure|Animation|Children|Comedy|Fantasy|War
